# Capítulo 12 — ¿Lo hace bien?

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Dirección pregunta: *«¿Qué tal funciona Nova Assist?»*. Solo tenemos impresiones y algún pantallazo.

| | |
|---|---|
| **🧠 Componente agéntico** | Observabilidad y evaluación |
| **La necesidad** | Ver cada decisión del agente y medir su calidad (y su coste) de forma sistemática |
| **Lo que construimos** | Primero un juez con rúbrica en local; después, para producción: **Observability** (trazas OTel), **Evaluations** online con muestreo e histórico en Iceberg |
| **🏠 En local** | ✅ Funciona a pequeña escala: un juez con rúbrica sobre unos pocos casos (sección 0). No escala a tráfico real ni sigue servicios distribuidos. |
| **☁️ Por qué AWS al salir a Internet** | Con `print()` y logs sueltos no se reconstruye una conversación con diez herramientas. Las trazas OTel siguen cada paso y las evaluaciones online con muestreo convierten la calidad en una métrica continua. |
| **En el ordenador sería…** | El **visor de eventos y los exámenes**: ver qué ha pasado y medir el rendimiento con pruebas repetibles |
| **Requisitos del caso** | RNF-05 |
| **Prerrequisito · duración** | Capítulo 11 · 60 min |
| **⚠️ Dónde se rompe la analogía** | El visor de eventos muestra lo que pasó; las evaluaciones con un LLM como juez **opinan** sobre lo que pasó. Hay que calibrar al juez. |

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(12)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(12)
# ponerse_al_dia(12)   # ← descomenta si llegas tarde o algo se rompió

## 🧠 La teoría: observabilidad y evaluación

- **Observabilidad:** poder reconstruir cada decisión del agente (trazas del modelo, las herramientas y la memoria).
- **Evaluación:** medir la calidad con **casos de referencia** (*golden set*) y **jueces** (código o un LLM con rúbrica).
  - *offline*, antes de desplegar;
  - *online*, sobre tráfico real y con muestreo.
- Es el **bucle de retroalimentación** del agente: sin medición no hay mejora.

## 0. El concepto en local: un juez con rúbrica

Evaluar un agente es comparar lo que hace con lo que debería hacer. En pequeño: tres casos de referencia, la respuesta del agente y un modelo que hace de **juez**.

In [ ]:
import json, re
from strands import Agent
from strands.models import BedrockModel

ds = json.loads((DATOS_DIR / "tiendanova.json").read_text(encoding="utf-8"))
casos_locales = [c for c in ds["eval_casos"] if c["caso_id"] in ("E01", "E07", "E10")]
RUBRICA = ("Eres un evaluador. Compara la respuesta del asistente con la respuesta esperada. Responde SOLO con JSON: "
           '{"correcta": true o false, "motivo": "una frase"}')
TOK = token_cliente("C-001")
for c in casos_locales:
    invocar(c["prompt"], token=TOK)
    juez = Agent(model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0),
                 system_prompt=RUBRICA, callback_handler=None)
    veredicto = str(juez(f"Pregunta: {c['prompt']}\nRespuesta esperada: {c['respuesta_esperada']}\n"
                         f"Respuesta del asistente: {ultima_respuesta()}"))
    m = re.search(r"\{.*\}", veredicto, re.S)
    print(f"⚖️ {c['caso_id']}:", json.loads(m.group(0)) if m else veredicto, "\n")

Funciona con tres casos y alguien mirando. En producción la situación es otra:
- hay **miles de conversaciones al día**;
- cada una atraviesa **varios servicios** (Runtime, Memory, Gateway, Lambdas, S3 Vectors);
- **nadie está mirando**.

Hace falta reconstruir cada decisión de principio a fin (**trazas distribuidas**), evaluar **tráfico real con muestreo** y guardar el **histórico**. Eso es lo que construye el resto del capítulo.

## 1. Observabilidad

Requisito: **CloudWatch Transaction Search** activado (capítulo 0). Abre después CloudWatch → *GenAI Observability* y recorre una sesión completa: modelo, herramientas del Gateway, Memory y Code Interpreter.

In [ ]:
!cd "{PROJECT}" && agentcore traces list

In [ ]:
!cd "{PROJECT}" && agentcore logs --since 30m --level error

## 2. Conjunto de prueba desde Iceberg

El conjunto de evaluación vive en la tabla Iceberg `eval_casos` (capítulo 0, Parte C): casos de referencia con herramientas y documento esperados, y casos adversariales. Ejecutamos los de los clientes con usuario en Cognito (C-001 y C-002).

In [ ]:
casos = [c for c in leer_tabla("eval_casos") if c["cliente"] in ("C-001", "C-002")]
tokens = {u: token_cliente(u) for u in ("C-001", "C-002")}
ejecuciones = []
for c in casos:
    print(f"— {c['caso_id']} [{c['categoria']}] esperado: {c['respuesta_esperada']}")
    sid = invocar(c["prompt"], token=tokens[c["cliente"]])
    ejecuciones.append({"caso_id": c["caso_id"], "session_id": sid})
state_set(EVAL_SESSIONS=[e["session_id"] for e in ejecuciones])

## 3. Evaluación bajo demanda

In [ ]:
for sid in state_get("EVAL_SESSIONS")[:3]:
    get_ipython().system(
        f'cd "{PROJECT}" && agentcore run eval --runtime NovaAgent --session-id {sid} '
        '--evaluator "Builtin.GoalSuccessRate" --evaluator "Builtin.Helpfulness"'
    )

Añade `Builtin.ToolSelectionAccuracy`, que trabaja a nivel de llamada a herramienta, para medir si el agente eligió la herramienta correcta. Si la evaluación no encuentra la sesión, espera unos segundos (las trazas tardan en estar disponibles) y repite.

## 4. Evaluación online continua

In [ ]:
# Los nombres de configuración usan guiones bajos. Configura el porcentaje de muestreo (--help).
!cd "{PROJECT}" && agentcore add online-eval --help

In [ ]:
!cd "{PROJECT}" && agentcore add online-eval --name nova_online_eval --evaluator "Builtin.GoalSuccessRate" "Builtin.Helpfulness"
!cd "{PROJECT}" && agentcore deploy --yes

Para pausarla y reanudarla: `agentcore pause online-eval nova_online_eval` y `agentcore resume online-eval nova_online_eval`.

> Evaluar el 100 % del tráfico con jueces LLM puede duplicar el consumo de inferencia (doc §8). **Muestrea.**

## 5. Evaluación del retrieval con histórico en Iceberg

El *recall@k* del capítulo 8 se convierte en una métrica con histórico: cada ejecución se añade a la tabla Iceberg `eval_retrieval`. Así puedes comparar modelos de *embeddings*, troceados o versiones de documentos a lo largo del tiempo.

In [ ]:
from datetime import datetime, timezone
import rag_nova

casos_rag = [c for c in leer_tabla("eval_casos") if c["documento_esperado"]]
ahora = datetime.now(timezone.utc).replace(tzinfo=None)
filas = []
for c in casos_rag:
    docs = [r["doc_id"] for r in rag_nova.buscar(c["prompt"], top_k=5)]
    filas.append({"ejecucion": ahora, "caso_id": c["caso_id"], "modelo_embedding": rag_nova.MODELO_EMBEDDING,
                  "documento_esperado": c["documento_esperado"], "top5": ",".join(docs),
                  "acierto_top1": docs[:1] == [c["documento_esperado"]],
                  "acierto_top3": c["documento_esperado"] in docs[:3]})
escribir_tabla("eval_retrieval", filas, modo="append")
for fila in athena("SELECT ejecucion, modelo_embedding, avg(CAST(acierto_top3 AS double)) AS recall_at_3 "
                   "FROM eval_retrieval GROUP BY ejecucion, modelo_embedding ORDER BY ejecucion"):
    print(fila)

## 6. Evaluador propio (opcional)

Crea un evaluador LLM "cumplimiento de la política de devoluciones" con criterios explícitos: cita el documento, no promete plazos fuera de la política y escala cuando procede. Añade *ground truth* con la secuencia de herramientas esperada para cada caso del conjunto de prueba.

## 7. Lo que revelan los datos

Además de evaluar al agente, la plataforma de datos permite mirar el **negocio**. Tasa de devolución por producto en los últimos 90 días:

In [ ]:
for fila in athena("""
    WITH entregas AS (
      SELECT l.producto_id, l.nombre, count(DISTINCT p.pedido_id) AS pedidos
      FROM pedidos p JOIN lineas_pedido l ON p.pedido_id = l.pedido_id
      WHERE p.estado = 'ENTREGADO' AND p.fecha_entrega >= current_date - interval '90' day
      GROUP BY 1, 2),
    dev AS (
      SELECT producto_id, count(*) AS devoluciones FROM devoluciones
      WHERE fecha >= current_date - interval '90' day GROUP BY 1)
    SELECT e.producto_id, e.nombre, e.pedidos, coalesce(d.devoluciones, 0) AS devoluciones,
           round(100.0 * coalesce(d.devoluciones, 0) / e.pedidos, 1) AS tasa_pct
    FROM entregas e LEFT JOIN dev d ON e.producto_id = d.producto_id
    ORDER BY tasa_pct DESC LIMIT 5"""):
    print(fila)

Un producto destaca muy por encima del resto. Guarda la pregunta: volverá en el capítulo 14.

In [ ]:
taximetro_historico()

## Para pensar

- ¿Qué umbral de `GoalSuccessRate` exigirías para pasar a producción?
- ¿Qué alarmas de CloudWatch crearías (errores, latencia, `ActiveSessionCount`, coste)?

## 🔎 Lo que acabamos de descubrir

Dos hallazgos. Los evaluadores señalan respuestas mejorables, pero **cambiar el prompt para todos los clientes da miedo**. Y los datos muestran un **pico de devoluciones del Robot aspirador S9**.

In [ ]:
deuda("D6", "pagada", 12, "coste medido por conversación y por capítulo; límites en el Gateway")
cuaderno()
taximetro_cerrar(12)
guardar_partida(12)

---
➡️ **Siguiente capítulo:** `13_ab_testing.ipynb` — *Mejorar sin romper*